# Baseline: TF-IDF + Logistic Regression (BANKING77)

Three feature sets with the same classifier, compared on the validation set. The official test set is not touched here: it is used once, for the final model.

In [ ]:
import random
import string

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import FeatureUnion, Pipeline

pd.set_option("display.max_colwidth", 120)

In [ ]:
# The MLflow server must be running in a separate terminal
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("banking77-intents")

In [ ]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

X_train, y_train = train["text"], train["intent"]
X_val, y_val = val["text"], val["intent"]
print(f"train={len(train)} val={len(val)} intents={y_train.nunique()}")

## How close is validation to train?

Share of validation requests with a near-duplicate in train (TF-IDF cosine). On Bitext this was 29.5% at ≥ 0.8.

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2)).fit(X_train)
nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(vec.transform(X_train))
dist, _ = nn.kneighbors(vec.transform(X_val))
sim = 1 - dist[:, 0]
for t in [0.8, 0.9, 0.95]:
    print(f"val texts with a train neighbour at cosine >= {t}: {(sim >= t).mean():.1%}")

## Feature sets

- **word**: word unigrams + bigrams
- **char**: character 2–5-grams inside word boundaries (robust to typos)
- **word+char**: both, concatenated

In [ ]:
def word_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)


def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)


feature_sets = {
    "word": word_tfidf(),
    "char": char_tfidf(),
    "word+char": FeatureUnion([("word", word_tfidf()), ("char", char_tfidf())]),
}

## Training

One MLflow run per feature set: parameters, validation metrics and the fitted pipeline.

In [ ]:
results, models = {}, {}
for name, features in feature_sets.items():
    with mlflow.start_run(run_name=f"baseline_logreg_{name}"):
        model = Pipeline([("features", features), ("clf", LogisticRegression(max_iter=3000))])
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        metrics = {
            "macro_f1": f1_score(y_val, pred, average="macro"),
            "accuracy": accuracy_score(y_val, pred),
        }

        mlflow.set_tag("stage", "baseline")
        mlflow.log_params(
            {"model": "logreg", "features": name, "C": 1.0, "train_rows": len(train)}
        )
        mlflow.log_metrics({f"val_{k}": v for k, v in metrics.items()})
        mlflow.sklearn.log_model(model, name="model")

    results[name], models[name] = metrics, model

results = pd.DataFrame(results).T.rename_axis("features").round(4)
results

## Error analysis

Most frequent confusions of the best feature set on the validation set.

In [ ]:
best_name = results["macro_f1"].idxmax()
best = models[best_name]
pred = best.predict(X_val)

labels = sorted(y_val.unique())
cm = pd.DataFrame(confusion_matrix(y_val, pred, labels=labels), index=labels, columns=labels)
cm.index.name, cm.columns.name = "true", "pred"

pairs = cm.stack().rename("count").reset_index()
errors_only = (pairs["true"] != pairs["pred"]) & (pairs["count"] > 0)
print(f"Best feature set: {best_name}, errors: {int((pred != y_val).sum())} of {len(y_val)}")
pairs[errors_only].sort_values("count", ascending=False).head(15)

In [ ]:
errors = val.assign(pred=pred).query("intent != pred")
errors[["instruction", "intent", "pred"]].sample(min(15, len(errors)), random_state=42)

## Confidence-based routing

Predictions below a threshold go to a human agent. Intents that trigger irreversible actions always require confirmation; here we count how often they are predicted wrongly.

In [ ]:
proba = best.predict_proba(X_val)
val_eval = val.assign(pred=pred, confidence=proba.max(axis=1), correct=pred == y_val.to_numpy())

rows = []
for t in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6]:
    auto = val_eval["confidence"] >= t
    rows.append({
        "threshold": t,
        "automated": auto.mean(),
        "accuracy_automated": val_eval.loc[auto, "correct"].mean() if auto.any() else float("nan"),
        "errors_automated": int((~val_eval.loc[auto, "correct"]).sum()),
    })
pd.DataFrame(rows).round(3)

In [ ]:
RISKY = {"terminate_account", "cancel_transfer", "lost_or_stolen_card", "compromised_card"}
critical = val_eval[val_eval["pred"].isin(RISKY) & ~val_eval["correct"]]
print(f"Critical errors (risky intent predicted wrongly): {len(critical)}")
critical[["instruction", "intent", "pred", "confidence"]]

## Robustness to synthetic typos

In [ ]:
def add_typos(text: str, rate: float, rng: random.Random) -> str:
    """Corrupt letters in alphabetic words: delete, swap, replace or insert, each char with prob `rate`."""
    words = []
    for word in text.split():
        if not word.isalpha():  # keep <num> and punctuation
            words.append(word)
            continue
        chars, out, i = list(word), [], 0
        while i < len(chars):
            if rng.random() < rate:
                op = rng.choice(["delete", "swap", "replace", "insert"])
                if op == "delete":
                    i += 1
                    continue
                if op == "swap" and i + 1 < len(chars):
                    out += [chars[i + 1], chars[i]]
                    i += 2
                    continue
                if op == "replace":
                    out.append(rng.choice(string.ascii_lowercase))
                    i += 1
                    continue
                if op == "insert":
                    out += [chars[i], rng.choice(string.ascii_lowercase)]
                    i += 1
                    continue
            out.append(chars[i])
            i += 1
        if out:
            words.append("".join(out))
    return " ".join(words)


rng = random.Random(42)
rows = []
for rate in [0.0, 0.05, 0.1, 0.2, 0.3]:
    noisy = X_val.map(lambda t: add_typos(t, rate, rng))
    row = {"typo_rate": rate}
    for name, model in models.items():
        row[name] = f1_score(y_val, model.predict(noisy), average="macro")
    rows.append(row)

robust = pd.DataFrame(rows).set_index("typo_rate").round(4)
robust

In [ ]:
ax = robust.plot(marker="o", figsize=(7, 4), title="Robustness to synthetic typos (validation)")
ax.set_xlabel("probability of a typo per character")
ax.set_ylabel("macro-F1")
plt.tight_layout()
plt.show()